# gpubench on a real GPU (Google Colab)

This notebook runs the full benchmark on a Colab GPU and produces the figures and `summary.md` used in the README.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
The full run takes roughly 10 minutes (the CPU baselines for large matrices are the slow part).

In [ ]:
!nvidia-smi

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/imalachowsk-creator/gpu-bench.git"  # <- change to your repository

!git clone -q $REPO_URL gpu-bench
%cd gpu-bench
!pip install -q -e ".[dev]"

In [ ]:
!python -m gpubench env

## 2. Tests, including the GPU-only ones

On Colab nothing is skipped: the GPU tests check correctness on CUDA, that cheats are caught on the GPU, the synchronisation pitfall, transfers and the memory model.

In [ ]:
!python -m pytest -v

## 3. Why `torch.cuda.synchronize()` matters

GPU kernels run asynchronously. Without a synchronisation point the timer stops when the work is *queued*, not when it is *done*. The "without sync" column is the kind of number a careless benchmark would publish.

In [ ]:
from gpubench.demo import sync_pitfall_demo
sync_pitfall_demo(sizes=(256, 1024, 2048, 4096)).round(3)

## 4. The anti-cheat guards

Each broken implementation below looks great on speed alone. The verifier must reject all of them and accept only the honest one.

In [ ]:
from gpubench.demo import cheat_demo, cheat_demo_passed
df = cheat_demo(backend_name="torch-cuda", size=2048)
display(df[["implementation", "time_median", "gflops", "status", "reasons"]])
print("All cheats caught:", cheat_demo_passed(df))

## 5. Full benchmark

In [ ]:
import torch
from gpubench.runner import SuiteConfig, run_suite
from gpubench.report import build_report

gpu_tag = torch.cuda.get_device_name(0).split()[-1].lower()   # e.g. "t4"
RUN_DIR = f"results/colab-{gpu_tag}"

cfg = SuiteConfig(preset="full", dtypes=["float32", "float16"])
run_suite(cfg, RUN_DIR)
summary_path = build_report(RUN_DIR)

## 6. Results

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

# Tables as Markdown (figures are shown in the next cell).
text = Path(summary_path).read_text()
display(Markdown("\n".join(l for l in text.splitlines() if not l.startswith("!["))))

In [ ]:
from IPython.display import Image
for name in ["time_vs_size", "speedup", "throughput", "transfer", "memory"]:
    p = Path(RUN_DIR) / "figures" / f"{name}.png"
    if p.exists():
        display(Image(filename=str(p)))

## 7. Save the results

Download the zip, unpack it into the `results/` folder of your repository and commit it, so the README can show real numbers from a real GPU.

In [ ]:
!zip -qr results.zip $RUN_DIR
from google.colab import files
files.download("results.zip")